# 04 · Another embedding model

Same model structures as the pipeline (`train.make_scorer`, `train.CONFIGS`), different encoder: does a stronger or
differently trained encoder help? The reference rows are the pipeline's own e5-small on the same snapshot.

Ideas: `intfloat/multilingual-e5-base` (278M, 768 dims), e5 with `white_prefix="passage: "` (retrieval style:
the black card asks, the white card answers), `BAAI/bge-m3` (568M, 1024 dims; ~2.3 GB of RAM, check free memory
first). The revision is looked up and pinned when `revision` is None.

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ml").is_dir() else Path.cwd().parent  # papermill: repo root; VS Code: notebooks/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

from ml import experiments as ex
from ml import lower_priority, train

print("CPU priority:", lower_priority())  # idle CPU only: the game comes first

In [ ]:
snapshot = None
hf_id = "intfloat/multilingual-e5-base"
dim = 768
revision = None
black_prefix = "query: "
white_prefix = "query: "
model_types = ["linear", "mlp"]
reference = "e5"        # a pipeline embedding to compare with on the same snapshot; None to skip
log_mlflow = True

In [ ]:
data = ex.load(snapshot)
print(data.snapshot.name, data.sizes())
black_model = ex.custom_model(hf_id, dim, black_prefix, revision)
white_model = ex.custom_model(hf_id, dim, white_prefix, black_model.revision)
print(black_model)
inputs, targets = data.tensors(black_model, white_model)

In [ ]:
results = ex.baselines(data)
if reference:
    ref_inputs, ref_targets = data.tensors(reference)
    for m in model_types:
        results.append(ex.fit(f"{m} {reference} (pipeline)", partial(train.make_scorer, 384, train.CONFIGS[m]),
                              ref_inputs, ref_targets, train.CONFIGS[m], {"model_type": m, "embedding": reference}))
for m in model_types:
    results.append(ex.fit(f"{m} {black_model.key}", partial(train.make_scorer, dim, train.CONFIGS[m]), inputs,
                          targets, train.CONFIGS[m],
                          {"model_type": m, "embedding": black_model.key, "embedding_hf_id": hf_id,
                           "embedding_revision": black_model.revision, "black_prefix": black_prefix,
                           "white_prefix": white_prefix}))
ex.table(results)

In [ ]:
if log_mlflow:
    print("MLflow parent run:", ex.log("04_other_encoder", data, results, {"hf_id": hf_id, "revision": black_model.revision, "black_prefix": black_prefix, "white_prefix": white_prefix}))